In [ ]:
import pandas as pd

# Load datasets
asv_counts = pd.read_csv("/content/tblcounts_asv_melt.csv")
taxonomy = pd.read_csv("/content/tblASVtaxonomy_silva132_v4v5_filter.csv")
clinical_data = pd.read_csv("/content/processed_dataset.csv")
asv_samples = pd.read_csv("/content/tblASVsamples.csv")

# Use transform to align the index with the original DataFrame
asv_counts['RelativeAbundance'] = asv_counts.groupby('SampleID')['Count'].transform(lambda x: x / x.sum())

# Step 2: Map ASVs to taxonomy
asv_data = pd.merge(asv_counts, taxonomy, on='ASV', how='left')

# Step 3: Aggregate ASVs at the Genus level
genus_data = asv_data.groupby(['SampleID', 'Genus'])['RelativeAbundance'].sum().reset_index()

# Step 4: Merge ASVs with sample metadata (tblASVSamples)
# Align SampleID with DayRelativeToNearestHCT and stool consistency
genus_data = pd.merge(genus_data, asv_samples[['SampleID', 'PatientID', 'DayRelativeToNearestHCT', 'Consistency']], on='SampleID', how='left')

# Step 5: Merge ASV features with clinical data
# Align clinical data with PatientID and DayRelativeToNearestHCT
final_dataset = pd.merge(clinical_data, genus_data, on=['PatientID', 'DayRelativeToNearestHCT'], how='left')

# Step 6: Handle missing data
# Fill missing ASV features with 0 (since some samples may lack certain taxa)
final_dataset.fillna(0, inplace=True)

# Step 7: Save the new dataset
final_dataset.to_csv("asv_interpretability_dataset.csv", index=False)
print("Final dataset created: 'asv_interpretability_dataset.csv'")

Final dataset created: 'asv_interpretability_dataset.csv'
